In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone

from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import execute_values

from utils.file_utils import FileUtils
from utils.ingestion import Ingestion

In [ ]:
load_dotenv(Path(__file__).resolve().parent.parent / ".env")

BASE_DIR = Path(__file__).resolve().parent.parent
SOURCE_DIR = BASE_DIR / "data" / "bronze" / "open_meteo" / "archive"

print("Archive source directory:", SOURCE_DIR)
print("Now (UTC):", datetime.now(timezone.utc).isoformat())

In [ ]:
def connect():
    return psycopg2.connect(
        host=os.getenv("POSTGRES_HOST"),
        port=os.getenv("POSTGRES_PORT"),
        dbname=os.getenv("POSTGRES_DATABASE"),
        user=os.getenv("POSTGRES_USERNAME"),
        password=os.getenv("POSTGRES_PASSWORD"),
    )


def parse_city(path: Path) -> str:
    return path.stem.rsplit("_", 1)[0]


def parse_ingestion_date(path: Path) -> str:
    return path.parent.name


def build_rows(path: Path) -> list[tuple]:
    raw = FileUtils.read_json(path)
    hourly = raw.get("hourly", {})
    times = hourly.get("time", [])
    if not isinstance(times, list):
        return []

    fields = [
        "temperature_2m",
        "apparent_temperature",
        "relative_humidity_2m",
        "dew_point_2m",
        "precipitation",
        "rain",
        "snowfall",
        "snow_depth",
        "weather_code",
        "wind_speed_10m",
        "wind_direction_10m",
    ]

    columns = [hourly.get(field, []) for field in fields]
    if any(not isinstance(col, list) for col in columns):
        return []
    if any(len(col) != len(times) for col in columns):
        return []

    city_name = parse_city(path)
    ingestion_date = parse_ingestion_date(path)
    rows = []
    for idx, observation_time in enumerate(times):
        rows.append(
            (
                city_name,
                ingestion_date,
                f"{observation_time}Z",
                columns[0][idx],
                columns[1][idx],
                columns[2][idx],
                columns[3][idx],
                columns[4][idx],
                columns[5][idx],
                columns[6][idx],
                columns[7][idx],
                columns[8][idx],
                columns[9][idx],
                columns[10][idx],
            )
        )
    return rows

In [ ]:
paths = sorted(SOURCE_DIR.rglob("*.json"))
print(f"Found {len(paths)} archive JSON files")

all_rows = []
skipped = []
for path in paths:
    rows = build_rows(path)
    if rows:
        all_rows.extend(rows)
    else:
        skipped.append(path)

print(f"Built {len(all_rows)} rows from archive files")
if skipped:
    print(f"Skipped {len(skipped)} invalid files")
    for path in skipped[:10]:
        print(path)

In [ ]:
insert_sql = """
INSERT INTO silver.archive_weather_hourly (
    city_name,
    ingestion_date,
    observation_time,
    temperature_2m,
    apparent_temperature,
    relative_humidity_2m,
    dew_point_2m,
    precipitation,
    rain,
    snowfall,
    snow_depth,
    weather_code,
    wind_speed_10m,
    wind_direction_10m
) VALUES %s
"""

with connect() as conn:
    with conn.cursor() as cursor:
        execute_values(cursor, insert_sql, all_rows)
    conn.commit()

print("Loaded rows into silver.archive_weather_hourly")